# Qwen2.5-VL-3B: ours vs every baseline at matched token budgets (4 test videos, Colab A100)

One model, the 4 locked test videos (204 questions), every row of the accuracy table, with the baselines set to
**ours' measured token budgets** instead of fixed 5% / 10%:

1. **Uniform K, Blind and Ours** (4 fovea ring settings) are scored first.
2. **Ours' budgets are measured** (visual tokens per question / Uniform K's) and every baseline's knob is set to give
   the same budget (`accuracy_tables/calibrate.py`): exact from frame counts for StreamingTOM, DyCoke, DTD, average
   pooling and the two-view GazeVLM, from the gaze for GazeVLM, from the encoded frames for PruneVid and FastVID.
   Only token counts are used, never answers, so nothing is tuned on the test questions' accuracy.
3. **The 8 baselines** are scored at those budgets (one condition per matched budget).
4. **Summary**: accuracy with 95% intervals (resampling videos) and the measured budget of every row.

Results go to `MyDrive/GazeVLM/accuracy_tables/qwen25_matched/` (separate from other runs) after every question;
rerun any cell to resume. Rough time on an A100: step 1 ~25 min, step 2 ~10 min, step 3 ~2.5-3 h.


In [ ]:
import os, sys, subprocess
import torch

# An A100 (bf16, 40 GB+) is needed: on a T4 (16 GB, no bf16) Uniform K's 96 frames run out of memory and
# the bf16 models crawl. Runtime > Change runtime type > A100 GPU.
gpu = torch.cuda.get_device_properties(0) if torch.cuda.is_available() else None
print('GPU:', f'{gpu.name}, {gpu.total_memory / 1e9:.0f} GB' if gpu else 'none')
assert gpu and torch.cuda.get_device_capability(0)[0] >= 8 and gpu.total_memory > 35e9, \
    'needs an A100 GPU runtime (bf16, >= 40 GB): Runtime > Change runtime type > A100'

REPO, BRANCH = '/content/GazeVLM', 'eyehead-gate'
URL = 'https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign.git'
if os.path.exists(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, URL, REPO], check=True)
os.chdir(REPO)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
# num2words: needed by SmolVLM2's processor (install before transformers is first imported)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'pytest', 'num2words', 'pyyaml'],
               check=True)

In [ ]:
import shutil
from google.colab import drive
drive.mount('/content/drive')

try:                                         # Hugging Face token from the Colab secret HF_TOKEN
    from google.colab import userdata
    from huggingface_hub import login
    if userdata.get('HF_TOKEN'):
        login(token=userdata.get('HF_TOKEN'))
except Exception as ex:
    print('no HF_TOKEN secret:', type(ex).__name__)

CONFIG = 'accuracy_tables/config.yaml'
if not os.path.exists(CONFIG):
    shutil.copy('accuracy_tables/config.example.yaml', CONFIG)
MODEL, VIDEOS = 'qwen2.5', 'test'
OUT = '/content/drive/MyDrive/GazeVLM/accuracy_tables/qwen25_matched'      # this run's results (on Drive)
MATCHED = f'{OUT}/config_matched.yaml'                                      # written by step 2, kept on Drive
POINTS = 2                   # match the baselines to ours' 2 smallest measured budgets (the 2 smallest fovea settings)
os.makedirs(OUT, exist_ok=True)
print('results ->', OUT)


In [ ]:
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_accuracy_tables.py', 'tests/test_calibrate.py',
                    'tests/test_gazevlm.py', 'tests/test_streamingtom.py', 'tests/test_dycoke.py', 'tests/test_fastvid.py',
                    'tests/test_prunevid.py', 'tests/test_dtd.py', 'tests/test_avgpool.py', 'tests/test_gazevlm2.py', '-q'],
                   capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0, 'unit tests failed'


## Step 1: Uniform K (100%), Blind (0%) and Ours

Parity against the stock model first (stops if it fails). Ours = the D+P gate's frames, each foveated at its gaze,
at the 4 ring settings of the config.


In [ ]:
get_ipython().system(f'python -m accuracy_tables.run --config {CONFIG} --model {MODEL} --rows uniform,blind,ours '
                     f'--videos {VIDEOS} --out-dir {OUT}')


## Step 2: measure ours' budgets and set every baseline to them

Prints ours' measured budget per fovea setting, picks the `POINTS` smallest, and writes `config_matched.yaml` (the
config with every baseline's knob set) and `qwen2.5/calibration.json` to `OUT`. To match other settings, use
`--match "R=70 a=160,R=180 a=40"` instead of `--points`.


In [ ]:
get_ipython().system(f'python -m accuracy_tables.calibrate --config {CONFIG} --model {MODEL} --points {POINTS} '
                     f'--videos {VIDEOS} --out-dir {OUT} --write {MATCHED}')
print(open(MATCHED).read()[:4000])


## Step 3: the 8 baselines at ours' budgets

GazeVLM (Chen & Qi), two-view GazeVLM, Uniform K + average pooling, StreamingTOM, DyCoke, FastVID, PruneVid, DTD, on the
Uniform K frames, one condition per matched budget (every knob is in the condition name). `--keep-going`: a failing
baseline does not stop the others; a pass / fail table is printed at the end.


In [ ]:
assert os.path.exists(MATCHED), 'run step 2 first'
get_ipython().system(f'python -m accuracy_tables.run --config {MATCHED} --model {MODEL} '
                     f'--rows gazevlm,gazevlm2,avgpool,streamingtom,dycoke,fastvid,prunevid,dtd --videos {VIDEOS} '
                     f'--out-dir {OUT} --keep-going')


## Step 4: summary

Accuracy with a 95% interval over videos, frames and visual tokens per question, and every row's measured token
budget vs Uniform K. Also written to `OUT/summary.csv`.


In [ ]:
get_ipython().system(f'python -m accuracy_tables.summarize --config {MATCHED} --videos {VIDEOS} --out-dir {OUT}')
